# Vegetation Detection Neural Network
### Sentinel-2 Imagery · Task 2
**Goal:** Train a neural network to detect vegetation pixels in Sentinel-2 RGB patches,  
using NDVI > 0.3 as the ground-truth label, with class balancing via weighted loss.

**Keynotes:** Be sure to use Python version--3.12.0 or else the program will not work

In [ ]:
# Install required libraries (run once)
import sys
!{sys.executable} -m pip install torch torchvision Pillow numpy matplotlib scikit-learn rasterio --quiet

In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, classification_report,
                              confusion_matrix)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("PyTorch version :", torch.__version__)
print("Device          :", "cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1 · Configuration
All paths and hyper-parameters in one place — easy to change.

In [ ]:
# ── Paths (mirrors Task 1 structure) ────────────────────────────────────────
ROUTE_SAMPLES = "data/samples"   # RGB  .tiff files  (_img_N)
ROUTE_LABELS  = "data/labels"    # NDVI .tiff files  (_ndvi_N)

# ── Hyper-parameters ─────────────────────────────────────────────────────────
NDVI_THRESHOLD = 0.3    # pixels above this → vegetation (class 1)
PATCH_SIZE     = 32     # NxN pixel patches fed to the network
STRIDE         = 8  # overlap between patches (data augmentation)
HIDDEN_SIZE    = 128    # neurons in the hidden layer
LEARNING_RATE  = 1e-3
EPOCHS         = 50
BATCH_SIZE     = 256
TEST_SPLIT     = 0.20   # 20 % held out for evaluation

print("Config loaded ✓")

## 2 · Data Loading & Preprocessing
We replicate the Task 1 reading logic and extend it:
- **RGB image** → 3 feature channels, normalised to [0, 1]
- **NDVI map** → normalised to [-1, 1], thresholded at 0.3 to create binary labels  
  - Class 0 = non-vegetation  
  - Class 1 = vegetation

In [ ]:
def load_pair(img_path: str, ndvi_path: str):
    """
    Load one (image, ndvi) pair exactly as Task 1 does, then return
    the normalised RGB array and the binary vegetation mask.
    """
    # ── RGB image ─────────────────────────────────────────────────────────────
    with Image.open(img_path) as im:
        rgb = np.array(im).astype(np.float32)          # (H, W, 3)

    # Normalise to [0, 1]  — same max-normalisation as Task 1
    max_val = rgb.max()
    if max_val > 0:
        rgb = rgb / max_val

    # ── NDVI label ───────────────────────────────────────────────────────────
    with Image.open(ndvi_path) as nm:
        ndvi = np.array(nm).astype(np.float32)         # (H, W)

    # Task 1 normalisation: rescale to [-1, 1]
    if ndvi.max() > 1 or ndvi.min() < -1:
        ndvi = (ndvi - ndvi.min()) / (ndvi.max() - ndvi.min()) * 2 - 1

    # Binary mask  (matches Task 1's   ndvi > 0.3  threshold)
    mask = (ndvi > NDVI_THRESHOLD).astype(np.int64)    # (H, W)  0 / 1

    return rgb, ndvi, mask


def discover_pairs(samples_dir: str, labels_dir: str):
    """Find all matched (img, ndvi) file pairs — same logic as Task 1."""
    files  = [f for f in os.listdir(samples_dir) if f.endswith(".tiff")]
    paired = [f for f in files
              if os.path.exists(os.path.join(labels_dir,
                                             f.replace("_img_", "_ndvi_")))]
    return [(os.path.join(samples_dir, f),
             os.path.join(labels_dir,  f.replace("_img_", "_ndvi_")))
            for f in sorted(paired)]


pairs = discover_pairs(ROUTE_SAMPLES, ROUTE_LABELS)
print(f"Found {len(pairs)} image pairs")

all_rgb, all_ndvi, all_mask = [], [], []
for img_p, ndvi_p in pairs:
    rgb, ndvi, mask = load_pair(img_p, ndvi_p)
    all_rgb.append(rgb); all_ndvi.append(ndvi); all_mask.append(mask)
    veg_pct = mask.mean() * 100
    print(f"  {os.path.basename(img_p)[-18:]}  "          f"shape={rgb.shape}  veg={veg_pct:.1f}%  non-veg={100-veg_pct:.1f}%")

print("\nData loaded ✓")

## 3 · Class Balance Analysis
Understanding the imbalance is essential before deciding how to fix it.

In [ ]:
fig, axes = plt.subplots(len(pairs), 4, figsize=(18, len(pairs) * 4.5))

total_veg = total_nonveg = 0

for i, (rgb, ndvi, mask) in enumerate(zip(all_rgb, all_ndvi, all_mask)):
    veg    = int(mask.sum())
    nonveg = int(mask.size - veg)
    total_veg    += veg
    total_nonveg += nonveg
    veg_pct = veg / mask.size * 100

    # RGB
    axes[i,0].imshow(rgb)
    axes[i,0].set_title(f"RGB — image {i}", fontsize=11)
    axes[i,0].axis("off")

    # NDVI heatmap
    im1 = axes[i,1].imshow(ndvi, cmap="RdYlGn", vmin=-1, vmax=1)
    axes[i,1].set_title(f"NDVI — {veg_pct:.1f}% vegetation", fontsize=11)
    axes[i,1].axis("off")
    plt.colorbar(im1, ax=axes[i,1], fraction=0.046, pad=0.04)

    # Binary mask
    axes[i,2].imshow(mask, cmap="Greens", vmin=0, vmax=1)
    axes[i,2].set_title("Ground truth mask", fontsize=11)
    axes[i,2].axis("off")
    veg_p   = mpatches.Patch(color="#2d8a4e", label=f"Vegetation ({veg_pct:.1f}%)")
    noveg_p = mpatches.Patch(color="#f0f0f0", label=f"Non-veg ({100-veg_pct:.1f}%)")
    axes[i,2].legend(handles=[veg_p, noveg_p], loc="lower right", fontsize=8)

    # Bar chart
    axes[i,3].bar(["Non-veg","Vegetation"], [nonveg, veg],
                  color=["#d9534f","#5cb85c"], edgecolor="white")
    axes[i,3].set_title("Pixel counts", fontsize=11)
    axes[i,3].set_ylabel("Pixels")
    for bar, val in zip(axes[i,3].patches, [nonveg, veg]):
        axes[i,3].text(bar.get_x()+bar.get_width()/2,
                       bar.get_height()+200, f"{val:,}",
                       ha="center", va="bottom", fontsize=9)

plt.suptitle("Class Balance Analysis — All Image Pairs", fontsize=14, y=1.01) 
plt.tight_layout()
plt.show()

overall_ratio = total_veg / (total_veg + total_nonveg) * 100
print(f"\nOverall  →  Vegetation: {total_veg:,} ({overall_ratio:.1f}%)  "      f"Non-veg: {total_nonveg:,} ({100-overall_ratio:.1f}%)")
print(f"Imbalance ratio: {max(total_veg,total_nonveg)/min(total_veg,total_nonveg):.2f}:1")

## 4 · Patch Dataset & Class Balancing

### Why patches?
Instead of feeding whole 256×256 images, we extract small **16×16 patches**.  
Each patch gives the network local spatial context — it sees neighbouring pixels,  
not just a single point. This is how CNNs work conceptually, but here we flatten  
the patch into a feature vector (16×16×3 = 768 features) for our MLP.

### How we balance the classes
Two complementary strategies:

| Strategy | Where | How |
|---|---|---|
| **WeightedRandomSampler** | DataLoader | Oversamples minority class during training |
| **Weighted CrossEntropyLoss** | Loss function | Penalises mistakes on the minority class more |

Together they prevent the network from just predicting "vegetation" for everything.

In [ ]:
class SentinelPatchDataset(Dataset):
    """
    Extracts NxN patches from all image pairs.
    Label = majority class of the patch centre pixel.
    """
    def __init__(self, rgb_list, mask_list, patch_size=PATCH_SIZE,
                 stride=STRIDE, augment=False):
        self.patches = []   # (C*P*P,) float32 tensors
        self.labels  = []   # int64 scalars

        half = patch_size // 2
        for rgb, mask in zip(rgb_list, mask_list):
            H, W = mask.shape
            for y in range(half, H - half, stride):
                for x in range(half, W - half, stride):
                    patch = rgb[y-half:y+half, x-half:x+half, :]  # (P,P,3)
                    label = int(mask[y, x])                         # centre px

                    feat = torch.tensor(patch.transpose(2,0,1),     # (3,P,P)
                                        dtype=torch.float32).flatten()
                    self.patches.append(feat)
                    self.labels.append(label)

                    # ── Augmentation (flip) for training set ─────────────
                    if augment:
                        for flipped in [
                            np.fliplr(patch),
                            np.flipud(patch),
                            np.rot90(patch),
                        ]:
                            f2 = torch.tensor(flipped.copy().transpose(2,0,1),
                                              dtype=torch.float32).flatten()
                            self.patches.append(f2)
                            self.labels.append(label)

        self.labels = torch.tensor(self.labels, dtype=torch.long)
        print(f"  Dataset size: {len(self.labels):,} patches  "              f"veg={self.labels.sum().item():,} "              f"({self.labels.float().mean()*100:.1f}%)")

    def __len__(self):  return len(self.labels)

    def __getitem__(self, idx):
        return self.patches[idx], self.labels[idx]


# ── Train / test split (by image, not by pixel — avoids data leakage) ────────
n = len(all_rgb)
n_test  = max(1, int(n * TEST_SPLIT))
n_train = n - n_test

print("Building training dataset  (with augmentation):")
train_ds = SentinelPatchDataset(all_rgb[:n_train], all_mask[:n_train], augment=True)
print("Building test dataset:")
test_ds  = SentinelPatchDataset(all_rgb[n_train:], all_mask[n_train:], augment=False)

INPUT_SIZE = train_ds.patches[0].shape[0]
print(f"\nInput feature size: {INPUT_SIZE}  ({PATCH_SIZE}×{PATCH_SIZE}×3)")

In [ ]:
# ── Class weights for WeightedRandomSampler ──────────────────────────────────
train_labels = train_ds.labels.numpy()
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts           # inverse-frequency weighting
sample_weights = class_weights[train_labels]  # weight for every sample

sampler = WeightedRandomSampler(
    weights     = torch.from_numpy(sample_weights).float(),
    num_samples = len(train_ds),
    replacement = True
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# ── Loss weights (complement to sampler) ─────────────────────────────────────
loss_weights = torch.tensor(class_weights / class_weights.sum(),
                             dtype=torch.float32).to(DEVICE) * 2

print(f"Class counts (train): Non-veg={class_counts[0]:,}  Veg={class_counts[1]:,}")
print(f"Loss weights        : Non-veg={loss_weights[0]:.4f}  Veg={loss_weights[1]:.4f}")
print(f"Train batches: {len(train_loader)}   Test batches: {len(test_loader)}")

## 5 · Neural Network Architecture

```
Input  (768)                 ← 16×16 patch × 3 RGB channels, flattened
  │
  ▼
Linear(768 → 128) + BN + ReLU + Dropout(0.3)    ← Hidden layer
  │
  ▼
Linear(128 → 2)                                  ← Output layer (logits)
  │
  ▼
CrossEntropyLoss  [applies softmax internally]
```

**ReLU**: f(z) = max(0, z)  — introduces nonlinearity so the network learns  
curves, not just straight-line decision boundaries.

**BatchNorm**: normalises each layer's activations during training — speeds up  
convergence and reduces sensitivity to the learning rate.

**Dropout**: randomly zeroes 30% of neurons per batch — prevents memorisation.

In [ ]:
class VegetationNet(nn.Module):
    """
    Input  → Hidden (BN + ReLU + Dropout) → Output
    Input size  = PATCH_SIZE * PATCH_SIZE * 3   (flattened patch)
    Output size = 2  (non-vegetation, vegetation)
    """
    def __init__(self, input_size: int, hidden_size: int = HIDDEN_SIZE):
        super().__init__()
        self.net = nn.Sequential(
            # ── Hidden layer ────────────────────────────────────────────────
            nn.Linear(input_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            # ── Output layer ────────────────────────────────────────────────
            nn.Linear(hidden_size, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


model     = VegetationNet(INPUT_SIZE, HIDDEN_SIZE).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=loss_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# ── Print architecture ────────────────────────────────────────────────────────
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

## 6 · Training

In [ ]:
history = {"loss": [], "train_acc": [], "test_acc": []}

print(f"{'Epoch':>6}  {'Loss':>8}  {'Train Acc':>10}  {'Test Acc':>9}")
print("-" * 42)

for epoch in range(1, EPOCHS + 1):

    # ── Training pass ─────────────────────────────────────────────────────────
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()                   # ← backpropagation (chain rule)
        optimizer.step()                  # ← gradient descent weight update
        epoch_loss += loss.item() * len(y_batch)

    scheduler.step()
    avg_loss = epoch_loss / len(train_ds)

    # ── Quick accuracy check every 5 epochs ───────────────────────────────────
    if epoch % 5 == 0 or epoch == 1:
        model.eval()
        def batch_acc(loader):
            correct = total = 0
            with torch.no_grad():
                for Xb, yb in loader:
                    preds    = model(Xb.to(DEVICE)).argmax(1).cpu()
                    correct += (preds == yb).sum().item()
                    total   += len(yb)
            return correct / total

        tr_acc = batch_acc(train_loader)
        te_acc = batch_acc(test_loader)
        history["loss"].append(avg_loss)
        history["train_acc"].append(tr_acc)
        history["test_acc"].append(te_acc)
        print(f"{epoch:>6}  {avg_loss:>8.4f}  {tr_acc*100:>9.2f}%  {te_acc*100:>9.2f}%")

print("\nTraining complete ✓")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

n = len(history["loss"])
epochs_logged = ([1] + list(range(5, EPOCHS+1, 5)))[:n]

ax1.plot(epochs_logged, history["loss"],
         color="#d9534f", linewidth=2, marker="o", markersize=5)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Cross-Entropy Loss")
ax1.set_title("Training Loss"); ax1.grid(alpha=0.3)

ax2.plot(epochs_logged, [v*100 for v in history["train_acc"]],
         label="Train", color="#5cb85c", linewidth=2, marker="o", markersize=5)
ax2.plot(epochs_logged, [v*100 for v in history["test_acc"]],
         label="Test",  color="#337ab7", linewidth=2, marker="s", markersize=5)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy Curves"); ax2.legend(); ax2.grid(alpha=0.3)
ax2.set_ylim([0, 100])

plt.tight_layout()
plt.show()

## 7 · Evaluation
We report the three key metrics **per class** and as macro averages:

| Metric | Formula | Meaning |
|---|---|---|
| **Accuracy** | (TP+TN) / N | Overall correct predictions |
| **Precision** | TP / (TP+FP) | Of predicted veg, how many are actually veg? |
| **Recall** | TP / (TP+FN) | Of actual veg pixels, how many did we catch? |

In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for Xb, yb in test_loader:
        preds = model(Xb.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_true.extend(yb.numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

acc  = accuracy_score(all_true, all_preds)
prec = precision_score(all_true, all_preds, average="macro", zero_division=0)
rec  = recall_score(all_true, all_preds, average="macro", zero_division=0)

print("=" * 50)
print("  Neural Network — Test Set Results")
print("=" * 50)
print(f"  Accuracy           : {acc*100:.2f}%")
print(f"  Precision (macro)  : {prec*100:.2f}%")
print(f"  Recall    (macro)  : {rec*100:.2f}%")
print()
print(classification_report(all_true, all_preds,
                             target_names=["Non-Vegetation", "Vegetation"],
                             digits=3))

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(all_true, all_preds)
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["Non-Veg","Vegetation"])
ax.set_yticklabels(["Non-Veg","Vegetation"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Test Set")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                color="white" if cm[i,j] > cm.max()/2 else "black", fontsize=12)
plt.tight_layout(); plt.show()

## 8 · Visualise Predictions on Full Images

In [ ]:
def predict_full_image(model, rgb, patch_size=PATCH_SIZE):
    """
    Slide a window over the full image and predict each centre pixel.
    Returns a (H, W) probability map for class 1 (vegetation).
    """
    model.eval()
    H, W = rgb.shape[:2]
    half  = patch_size // 2
    prob_map = np.zeros((H, W), dtype=np.float32)

    patches, coords = [], []
    for y in range(half, H - half):
        for x in range(half, W - half):
            patch = rgb[y-half:y+half, x-half:x+half, :]
            feat  = torch.tensor(patch.transpose(2,0,1),
                                  dtype=torch.float32).flatten()
            patches.append(feat)
            coords.append((y, x))
            if len(patches) == 512:          # mini-batch for speed
                with torch.no_grad():
                    logits = model(torch.stack(patches).to(DEVICE))
                    probs  = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
                for (py, px), p in zip(coords, probs):
                    prob_map[py, px] = p
                patches, coords = [], []

    if patches:                              # remaining
        with torch.no_grad():
            logits = model(torch.stack(patches).to(DEVICE))
            probs  = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
        for (py, px), p in zip(coords, probs):
            prob_map[py, px] = p

    return prob_map


# ── Plot all images ───────────────────────────────────────────────────────────
n_imgs = len(all_rgb)
fig, axes = plt.subplots(n_imgs, 4, figsize=(20, n_imgs * 5))
if n_imgs == 1: axes = axes[np.newaxis, :]

for i, (rgb, ndvi, mask) in enumerate(zip(all_rgb, all_ndvi, all_mask)):
    print(f"Predicting image {i}...", end=" ", flush=True)
    prob_map   = predict_full_image(model, rgb)
    pred_mask  = (prob_map > 0.5).astype(int)
    print("done")

    axes[i,0].imshow(rgb)
    axes[i,0].set_title(f"RGB — image {i}", fontsize=11)
    axes[i,0].axis("off")

    im1 = axes[i,1].imshow(ndvi, cmap="RdYlGn", vmin=-1, vmax=1)
    axes[i,1].set_title("NDVI ground truth", fontsize=11)
    axes[i,1].axis("off")
    plt.colorbar(im1, ax=axes[i,1], fraction=0.046, pad=0.04)

    im2 = axes[i,2].imshow(prob_map, cmap="YlGn", vmin=0, vmax=1)
    axes[i,2].set_title("NN probability map", fontsize=11)
    axes[i,2].axis("off")
    plt.colorbar(im2, ax=axes[i,2], fraction=0.046, pad=0.04)

    # overlay: green=correct, red=missed veg, orange=false veg
    overlay = np.zeros((*mask.shape, 3), dtype=np.float32)
    overlay[(mask==1)&(pred_mask==1)] = [0.2, 0.8, 0.2]   # TP green
    overlay[(mask==1)&(pred_mask==0)] = [0.9, 0.1, 0.1]   # FN red
    overlay[(mask==0)&(pred_mask==1)] = [1.0, 0.6, 0.0]   # FP orange
    overlay[(mask==0)&(pred_mask==0)] = [0.9, 0.9, 0.9]   # TN gray
    axes[i,3].imshow(overlay)
    axes[i,3].set_title("Prediction overlay", fontsize=11)
    axes[i,3].axis("off")
    tp = mpatches.Patch(color="#33cc33", label="True positive (veg caught)")
    fn = mpatches.Patch(color="#cc1a1a", label="False negative (veg missed)")
    fp = mpatches.Patch(color="#ff9900", label="False positive (wrong veg)")
    tn = mpatches.Patch(color="#e5e5e5", label="True negative")
    axes[i,3].legend(handles=[tp,fn,fp,tn], loc="lower right", fontsize=7)

plt.suptitle("Vegetation Detection — Full Image Predictions",
             fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

## 9 · Save the Model

In [ ]:
torch.save({
    "model_state":  model.state_dict(),
    "input_size":   INPUT_SIZE,
    "hidden_size":  HIDDEN_SIZE,
    "patch_size":   PATCH_SIZE,
    "threshold":    NDVI_THRESHOLD,
}, "vegetation_nn.pt")

print("Model saved → vegetation_nn.pt")
print("\nTo reload later:")
print("  ckpt  = torch.load('vegetation_nn.pt')")
print("  model = VegetationNet(ckpt['input_size'], ckpt['hidden_size'])")
print("  model.load_state_dict(ckpt['model_state'])")